<a href="https://colab.research.google.com/github/morenojosedev/Prueba-Tecnica-Qaroni/blob/main/PruebaTecnica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INSTALAR PYSPARK

In [1]:
!pip install pyspark

## CREAR SESION PYSPARK

In [2]:
from pyspark.sql import SparkSession
from pyspark import SparkContext

SpSession = SparkSession \
              .builder \
              .appName ("Prueba Tecnica") \
              .getOrCreate()

In [3]:
Spcontext = SpSession.sparkContext

# **CARGA DE DATOS Y TRANSFORMACIÓN**

Carga de datos de empresas sostenibilidad

In [5]:
import subprocess


subprocess.run(["wget", "-q",
    "https://raw.githubusercontent.com/morenojosedev/Prueba-Tecnica-Qaroni/refs/heads/main/data/input/Empresas_Sostenibilidad.csv",
    "-O", "/content/Empresas_Sostenibilidad.csv"])


dataEmpresasSos = SpSession.read.csv(
    "/content/Empresas_Sostenibilidad.csv",
    header=True,
    sep=","
)
dataEmpresasSos.show(5)

+----------+---------+-----------+-------+---------------+-------------+----------------------+
|empresa_id|   nombre|     sector|   pais|consumo_energia|emisiones_co2|certificacion_iso14001|
+----------+---------+-----------+-------+---------------+-------------+----------------------+
| EMP000000|Empresa_0|   EnergÃ­a|CanadÃ¡|        44404.7|       3217.3|                     0|
| EMP000001|Empresa_1|   Finanzas|  China|       25576.11|      16135.7|                     1|
| EMP000002|Empresa_2|      Salud| JapÃ³n|       29305.07|      2224.93|                     0|
| EMP000003|Empresa_3| Transporte|  China|       44600.25|      3764.32|                     1|
| EMP000004|Empresa_4|TecnologÃ­a|Francia|       46039.63|     11840.17|                     1|
+----------+---------+-----------+-------+---------------+-------------+----------------------+
only showing top 5 rows


 EXPLORANDO LA DATA

In [6]:
dataEmpresasSos.printSchema()

root
 |-- empresa_id: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- pais: string (nullable = true)
 |-- consumo_energia: string (nullable = true)
 |-- emisiones_co2: string (nullable = true)
 |-- certificacion_iso14001: string (nullable = true)



TRANSFORMACIÓN DE DATOS

In [7]:
from pyspark.sql.functions import col, sum as _sum, when, coalesce, lit
from pyspark.sql.types import DoubleType,IntegerType

empresas_transformado = dataEmpresasSos \
    .withColumn("emisiones_co2", col("emisiones_co2").cast(DoubleType())) \
    .withColumn("consumo_energia", col("consumo_energia").cast(DoubleType())) \
    .withColumn("certificacion_iso14001", col("certificacion_iso14001").cast(IntegerType()))

In [8]:
empresas_transformado.printSchema()

root
 |-- empresa_id: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- pais: string (nullable = true)
 |-- consumo_energia: double (nullable = true)
 |-- emisiones_co2: double (nullable = true)
 |-- certificacion_iso14001: integer (nullable = true)



Verificar que no tenga Nulls

In [9]:
from pyspark.sql.functions import col, sum, when


conteo_nulosEmpresa = [sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in empresas_transformado.columns]


empresas_transformado.select(conteo_nulosEmpresa).show()

+----------+------+------+----+---------------+-------------+----------------------+
|empresa_id|nombre|sector|pais|consumo_energia|emisiones_co2|certificacion_iso14001|
+----------+------+------+----+---------------+-------------+----------------------+
|         0|     0|     0|   0|              0|            0|                     0|
+----------+------+------+----+---------------+-------------+----------------------+



 Carga datos proyectos energeticos

In [10]:


subprocess.run(["wget", "-q",
    "https://raw.githubusercontent.com/morenojosedev/Prueba-Tecnica-Qaroni/refs/heads/main/data/input/Proyectos_Energeticos.csv",
    "-O", "/content/Proyectos_Energeticos.csv"])

# Ahora leer desde la ruta local
dataProyectos = SpSession.read.csv(
    "/content/Proyectos_Energeticos.csv",
    header=True,
    sep=","
)
dataProyectos.show(5)



+-----------+----------+------------+--------------------+-------------------+--------------+---------------+
|proyecto_id|empresa_id|tipo_energia|capacidad_generacion|reduccion_emisiones|costo_proyecto|estado_proyecto|
+-----------+----------+------------+--------------------+-------------------+--------------+---------------+
| PROJ000000| EMP016520|       Solar|             9131.06|            2270.88|         26.03|         Activo|
| PROJ000001| EMP012923|       Solar|             2164.32|             4886.1|        364.36|     Finalizado|
| PROJ000002| EMP023136|     Biomasa|             2374.24|              446.3|        190.17|  En desarrollo|
| PROJ000003| EMP004436| GeotÃ©rmica|             4065.11|            4465.83|         126.6|     Finalizado|
| PROJ000004| EMP021681|     EÃ³lica|             4581.51|            3280.46|        227.13|         Activo|
+-----------+----------+------------+--------------------+-------------------+--------------+---------------+
only showi

 EXPLORANDO LA DATA DE PROYECTOS

In [11]:
dataProyectos.printSchema()

root
 |-- proyecto_id: string (nullable = true)
 |-- empresa_id: string (nullable = true)
 |-- tipo_energia: string (nullable = true)
 |-- capacidad_generacion: string (nullable = true)
 |-- reduccion_emisiones: string (nullable = true)
 |-- costo_proyecto: string (nullable = true)
 |-- estado_proyecto: string (nullable = true)



 Transformamos los datos

In [12]:
dataProyectos_transformado = dataProyectos \
    .withColumn("capacidad_generacion", col("capacidad_generacion").cast(DoubleType())) \
    .withColumn("reduccion_emisiones", col("reduccion_emisiones").cast(DoubleType())) \
    .withColumn("costo_proyecto", col("costo_proyecto").cast(DoubleType()))

In [13]:
dataProyectos_transformado.printSchema()

root
 |-- proyecto_id: string (nullable = true)
 |-- empresa_id: string (nullable = true)
 |-- tipo_energia: string (nullable = true)
 |-- capacidad_generacion: double (nullable = true)
 |-- reduccion_emisiones: double (nullable = true)
 |-- costo_proyecto: double (nullable = true)
 |-- estado_proyecto: string (nullable = true)



 VERIFICAMOS QUE NO TENGA CASILLAS VACIAS

In [14]:
conteo_nulosProyectos = [sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in dataProyectos_transformado.columns]


dataProyectos_transformado.select(conteo_nulosProyectos).show()

+-----------+----------+------------+--------------------+-------------------+--------------+---------------+
|proyecto_id|empresa_id|tipo_energia|capacidad_generacion|reduccion_emisiones|costo_proyecto|estado_proyecto|
+-----------+----------+------------+--------------------+-------------------+--------------+---------------+
|          0|         0|           0|                   0|                  0|             0|              0|
+-----------+----------+------------+--------------------+-------------------+--------------+---------------+



 CARGA DATOS REGULACIONES AMBIENTALES

In [15]:

subprocess.run(["wget", "-q",
    "https://raw.githubusercontent.com/morenojosedev/Prueba-Tecnica-Qaroni/refs/heads/main/data/input/Regulaciones_Ambientales.csv",
    "-O", "/content/Regulaciones_Ambientales.csv"])

dataRegulaciones = SpSession.read.csv(
    "/content/Regulaciones_Ambientales.csv",
    header=True,
    sep=","
)
dataRegulaciones.show(5)


+-------------+--------+----------------+--------------------+----------------+
|regulacion_id|    pais|limite_emisiones|subsidios_renovables|impuesto_carbono|
+-------------+--------+----------------+--------------------+----------------+
|    REG000000|   India|         3012.63|                   1|           19.54|
|    REG000001|    EEUU|         7325.54|                   1|           42.41|
|    REG000002|Alemania|         9060.84|                   1|            6.15|
|    REG000003| EspaÃ±a|         7075.95|                   1|           44.65|
|    REG000004|Alemania|         1997.42|                   0|            8.75|
+-------------+--------+----------------+--------------------+----------------+
only showing top 5 rows


 EXPLORANDO LA DATA

In [16]:
dataRegulaciones.printSchema()

root
 |-- regulacion_id: string (nullable = true)
 |-- pais: string (nullable = true)
 |-- limite_emisiones: string (nullable = true)
 |-- subsidios_renovables: string (nullable = true)
 |-- impuesto_carbono: string (nullable = true)



Transformamos los datos

In [17]:
regulaciones_transformado = dataRegulaciones \
                            .withColumn("limite_emisiones", col("limite_emisiones").cast(DoubleType())) \
                            .withColumn("subsidios_renovables", col ("subsidios_renovables").cast(IntegerType())) \
                            .withColumn("impuesto_carbono", col("impuesto_carbono").cast(DoubleType()))

In [18]:
from pyspark.sql.functions import avg, first, round

regulaciones_agrupadas = regulaciones_transformado.groupBy("pais").agg(
    first("regulacion_id").alias("regulacion_id"),
   round( avg("limite_emisiones"),2).alias("limite_emisiones"),
   round( avg("impuesto_carbono"),2).alias("impuesto_carbono"))

In [19]:
regulaciones_agrupadas.show()

+--------+-------------+----------------+----------------+
|    pais|regulacion_id|limite_emisiones|impuesto_carbono|
+--------+-------------+----------------+----------------+
|Alemania|    REG000002|         7978.54|            25.3|
|  Brasil|    REG000010|         8143.87|           25.75|
| CanadÃ¡|    REG000005|         7890.71|           25.86|
|   China|    REG000019|         7973.63|           25.85|
|    EEUU|    REG000001|         7888.13|           25.77|
| EspaÃ±a|    REG000003|         8060.64|           25.77|
| Francia|    REG000008|          8037.0|           25.42|
|   India|    REG000000|         8073.86|           25.56|
|  JapÃ³n|    REG000016|         8063.89|           26.01|
| MÃ©xico|    REG000011|         7950.01|           25.48|
+--------+-------------+----------------+----------------+



Verificamos casillas vacias

In [20]:
conteo_nulosRegulacion = [sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in regulaciones_transformado.columns]


regulaciones_transformado.select(conteo_nulosRegulacion).show()

+-------------+----+----------------+--------------------+----------------+
|regulacion_id|pais|limite_emisiones|subsidios_renovables|impuesto_carbono|
+-------------+----+----------------+--------------------+----------------+
|            0|   0|               0|                   0|               0|
+-------------+----+----------------+--------------------+----------------+



# **Tabla de Salida 1: Impacto Ambiental Empresas**

Agrupacion por empresas sumando la reducciones de emisiones

In [21]:
proyectos_agrupados = dataProyectos_transformado.groupBy("empresa_id").agg(
    round(_sum("reduccion_emisiones"),2).alias("total_reduccion")
)

In [22]:
proyectos_agrupados.printSchema()

root
 |-- empresa_id: string (nullable = true)
 |-- total_reduccion: double (nullable = true)



UNION DE EMPRESAS CON LOS PROYECTOS AGRUPADOS

In [23]:
df_unido = empresas_transformado.join(proyectos_agrupados, on="empresa_id", how="left")

In [24]:
df_unido.show()

+----------+----------+-----------+-------+---------------+-------------+----------------------+---------------+
|empresa_id|    nombre|     sector|   pais|consumo_energia|emisiones_co2|certificacion_iso14001|total_reduccion|
+----------+----------+-----------+-------+---------------+-------------+----------------------+---------------+
| EMP000000| Empresa_0|   EnergÃ­a|CanadÃ¡|        44404.7|       3217.3|                     0|         831.24|
| EMP000001| Empresa_1|   Finanzas|  China|       25576.11|      16135.7|                     1|        1565.71|
| EMP000002| Empresa_2|      Salud| JapÃ³n|       29305.07|      2224.93|                     0|           NULL|
| EMP000003| Empresa_3| Transporte|  China|       44600.25|      3764.32|                     1|           NULL|
| EMP000004| Empresa_4|TecnologÃ­a|Francia|       46039.63|     11840.17|                     1|        3963.83|
| EMP000005| Empresa_5|Manufactura|CanadÃ¡|       41988.85|      5665.54|                     1|

In [25]:
df_unido_limpio = df_unido.fillna({"total_reduccion": 0.0})

In [26]:
df_unido_limpio.show(10)

+----------+---------+-----------+-------+---------------+-------------+----------------------+---------------+
|empresa_id|   nombre|     sector|   pais|consumo_energia|emisiones_co2|certificacion_iso14001|total_reduccion|
+----------+---------+-----------+-------+---------------+-------------+----------------------+---------------+
| EMP000000|Empresa_0|   EnergÃ­a|CanadÃ¡|        44404.7|       3217.3|                     0|         831.24|
| EMP000001|Empresa_1|   Finanzas|  China|       25576.11|      16135.7|                     1|        1565.71|
| EMP000002|Empresa_2|      Salud| JapÃ³n|       29305.07|      2224.93|                     0|            0.0|
| EMP000003|Empresa_3| Transporte|  China|       44600.25|      3764.32|                     1|            0.0|
| EMP000004|Empresa_4|TecnologÃ­a|Francia|       46039.63|     11840.17|                     1|        3963.83|
| EMP000005|Empresa_5|Manufactura|CanadÃ¡|       41988.85|      5665.54|                     1|        3

In [27]:
df_completo = df_unido_limpio.join(regulaciones_agrupadas, on="pais", how="left")

In [28]:
df_completo.show(20)

+-------+----------+----------+-----------+---------------+-------------+----------------------+---------------+-------------+----------------+----------------+
|   pais|empresa_id|    nombre|     sector|consumo_energia|emisiones_co2|certificacion_iso14001|total_reduccion|regulacion_id|limite_emisiones|impuesto_carbono|
+-------+----------+----------+-----------+---------------+-------------+----------------------+---------------+-------------+----------------+----------------+
|CanadÃ¡| EMP000000| Empresa_0|   EnergÃ­a|        44404.7|       3217.3|                     0|         831.24|    REG000005|         7890.71|           25.86|
|  China| EMP000001| Empresa_1|   Finanzas|       25576.11|      16135.7|                     1|        1565.71|    REG000019|         7973.63|           25.85|
| JapÃ³n| EMP000002| Empresa_2|      Salud|       29305.07|      2224.93|                     0|            0.0|    REG000016|         8063.89|           26.01|
|  China| EMP000003| Empresa_3| Tr

**Cálculo columnas requeridas para la reglas del negocio**

●	emisiones_netas
Cantidad final de emisiones que genera una empresa después de aplicar las reducciones asociadas a sus proyectos o actividades de mitigación.
Corresponde al valor de emisiones de la empresa menos la suma de todas las reducciones de emisiones registradas para esa misma empresa.




In [29]:
df_resultado = df_completo.withColumn("emisiones_netas", round( col("emisiones_co2") - coalesce(col("total_reduccion"),lit(0.0)),2))

In [30]:
df_resultado.show(5)

+-------+----------+---------+-----------+---------------+-------------+----------------------+---------------+-------------+----------------+----------------+---------------+
|   pais|empresa_id|   nombre|     sector|consumo_energia|emisiones_co2|certificacion_iso14001|total_reduccion|regulacion_id|limite_emisiones|impuesto_carbono|emisiones_netas|
+-------+----------+---------+-----------+---------------+-------------+----------------------+---------------+-------------+----------------+----------------+---------------+
|CanadÃ¡| EMP000000|Empresa_0|   EnergÃ­a|        44404.7|       3217.3|                     0|         831.24|    REG000005|         7890.71|           25.86|        2386.06|
|  China| EMP000001|Empresa_1|   Finanzas|       25576.11|      16135.7|                     1|        1565.71|    REG000019|         7973.63|           25.85|       14569.99|
| JapÃ³n| EMP000002|Empresa_2|      Salud|       29305.07|      2224.93|                     0|            0.0|    REG00

●	cumple_normativa
Indicador que señala si la empresa respeta el límite de emisiones establecido por la regulación que le aplica.
Se considera que una empresa cumple cuando sus emisiones netas son menores que el límite definido en la regulación correspondiente. En caso contrario, no cumple.


In [31]:
df_resultado = df_resultado.withColumn("cumple_Normativa", when(col("emisiones_netas")< col("limite_emisiones"),1).otherwise(0))

●	costo_impuesto
Costo estimado que la empresa tendría que pagar en caso de superar el límite permitido por la regulación.
Se calcula a partir del exceso de emisiones sobre el límite aplicado al valor del impuesto asociado a la regulación correspondiente.


In [32]:
df_resultado = df_resultado.withColumn(
    "costo_impuesto",
    when(
        col("emisiones_netas") > col("limite_emisiones"),
        round((col("emisiones_netas") - col("limite_emisiones")) * col("impuesto_carbono"), 2)
    ).otherwise(0.0)
)

TABLA SALIDA

In [33]:
tabla_salida_1 = df_resultado.select(
    "empresa_id",
    "nombre",
    "emisiones_netas",
    "cumple_normativa",
    "regulacion_id",
    "costo_impuesto"
)

In [34]:
tabla_salida_1.show()

+----------+----------+---------------+----------------+-------------+--------------+
|empresa_id|    nombre|emisiones_netas|cumple_normativa|regulacion_id|costo_impuesto|
+----------+----------+---------------+----------------+-------------+--------------+
| EMP000000| Empresa_0|        2386.06|               1|    REG000005|           0.0|
| EMP000001| Empresa_1|       14569.99|               0|    REG000019|     170515.91|
| EMP000002| Empresa_2|        2224.93|               1|    REG000016|           0.0|
| EMP000003| Empresa_3|        3764.32|               1|    REG000019|           0.0|
| EMP000004| Empresa_4|        7876.34|               1|    REG000008|           0.0|
| EMP000005| Empresa_5|        2147.87|               1|    REG000005|           0.0|
| EMP000006| Empresa_6|        3871.34|               1|    REG000019|           0.0|
| EMP000007| Empresa_7|        6224.86|               1|    REG000010|           0.0|
| EMP000008| Empresa_8|       10384.66|               

 EXPORTAMOS TABLA SALIDA 1

In [35]:
import os
import glob

tabla_salida_1.coalesce(1).write.mode("overwrite").options(header="True", sep=",").csv("tabla_salida_1")


archivos_part = glob.glob("tabla_salida_1/part-*.csv")

if archivos_part:

    archivo_origen = archivos_part[0]
    archivo_destino = "tabla_salida_1.csv"


    os.rename(archivo_origen, archivo_destino)
    print(f"¡Listo! Tu archivo se guardó y renombró correctamente como: {archivo_destino}")
else:
    print("No se encontró el archivo part dentro de la carpeta.")



¡Listo! Tu archivo se guardó y renombró correctamente como: tabla_salida_1.csv


# **Tabla de Salida 2: Beneficios Proyectos Energéticos**

●	total_inversion
Monto total invertido por una empresa en proyectos del mismo tipo de energía. Corresponde a la suma de los costos de todos los proyectos que comparten empresa y tipo de energía.


In [36]:
dataProyectos_transformado.printSchema()

root
 |-- proyecto_id: string (nullable = true)
 |-- empresa_id: string (nullable = true)
 |-- tipo_energia: string (nullable = true)
 |-- capacidad_generacion: double (nullable = true)
 |-- reduccion_emisiones: double (nullable = true)
 |-- costo_proyecto: double (nullable = true)
 |-- estado_proyecto: string (nullable = true)



In [37]:
dfProyectos = dataProyectos_transformado.groupBy(
    "empresa_id","tipo_energia").agg(
        round(_sum("costo_proyecto"),2).alias("total_inversion"))

In [38]:
dfProyectos.show()

+----------+---------------+---------------+
|empresa_id|   tipo_energia|total_inversion|
+----------+---------------+---------------+
| EMP018814|          Solar|          104.5|
| EMP013336|    GeotÃ©rmica|         317.44|
| EMP002480|        EÃ³lica|         234.03|
| EMP024299|        Biomasa|         375.14|
| EMP012636|    GeotÃ©rmica|         140.51|
| EMP005264|          Solar|          53.65|
| EMP005334|        EÃ³lica|          228.1|
| EMP010342|        Biomasa|          41.02|
| EMP003920|        Biomasa|          357.7|
| EMP003511|HidroelÃ©ctrica|         341.48|
| EMP024615|        EÃ³lica|         143.14|
| EMP017755|          Solar|          27.11|
| EMP004039|    GeotÃ©rmica|          89.24|
| EMP018584|        Biomasa|         330.54|
| EMP008283|HidroelÃ©ctrica|         420.82|
| EMP005895|          Solar|         662.98|
| EMP002865|    GeotÃ©rmica|         569.15|
| EMP002705|          Solar|          99.57|
| EMP010221|          Solar|         429.65|
| EMP02083

In [39]:
dfProyectosLimpio =  dfProyectos.fillna({"total_inversion" : 0.0})

In [40]:
dfProyectosLimpio.orderBy("total_inversion", ascending= True).show()

+----------+---------------+---------------+
|empresa_id|   tipo_energia|total_inversion|
+----------+---------------+---------------+
| EMP005305|HidroelÃ©ctrica|           0.11|
| EMP002840|        EÃ³lica|           0.13|
| EMP009997|        EÃ³lica|           0.16|
| EMP010754|          Solar|            0.2|
| EMP016726|    GeotÃ©rmica|           0.21|
| EMP022417|        EÃ³lica|           0.35|
| EMP019691|HidroelÃ©ctrica|           0.41|
| EMP013946|HidroelÃ©ctrica|           0.41|
| EMP006189|        EÃ³lica|           0.45|
| EMP013932|        Biomasa|           0.47|
| EMP011458|HidroelÃ©ctrica|           0.49|
| EMP023210|        EÃ³lica|           0.53|
| EMP005585|    GeotÃ©rmica|           0.55|
| EMP006376|          Solar|           0.63|
| EMP001349|        Biomasa|           0.64|
| EMP000642|    GeotÃ©rmica|           0.67|
| EMP020089|HidroelÃ©ctrica|           0.68|
| EMP005568|    GeotÃ©rmica|           0.68|
| EMP002921|        EÃ³lica|            0.7|
| EMP00908

In [41]:
df_proyectosUnidos = dataProyectos_transformado.join(dfProyectosLimpio,on=["empresa_id", "tipo_energia"],how="left")

In [42]:
df_proyectosUnidos.show(20)

+----------+---------------+-----------+--------------------+-------------------+--------------+---------------+---------------+
|empresa_id|   tipo_energia|proyecto_id|capacidad_generacion|reduccion_emisiones|costo_proyecto|estado_proyecto|total_inversion|
+----------+---------------+-----------+--------------------+-------------------+--------------+---------------+---------------+
| EMP016520|          Solar| PROJ000000|             9131.06|            2270.88|         26.03|         Activo|          26.03|
| EMP012923|          Solar| PROJ000001|             2164.32|             4886.1|        364.36|     Finalizado|         386.61|
| EMP023136|        Biomasa| PROJ000002|             2374.24|              446.3|        190.17|  En desarrollo|         190.17|
| EMP004436|    GeotÃ©rmica| PROJ000003|             4065.11|            4465.83|         126.6|     Finalizado|          126.6|
| EMP021681|        EÃ³lica| PROJ000004|             4581.51|            3280.46|        227.13| 

●	energia_generada_total
Capacidad total de generación asociada a los proyectos de una empresa para un tipo de energía específico. Se calcula como la suma de la energía generada por todos los proyectos de esa categoría.


In [43]:
dataProyectos_energia = dataProyectos_transformado.withColumn(
    "capacidad_generacion", col("capacidad_generacion").cast(DoubleType())
)

df_energia_agrupada = dataProyectos_energia .groupBy("empresa_id", "tipo_energia").agg(
    round(_sum("capacidad_generacion"), 2).alias("energia_generada_total")
)

In [44]:
dfProyectos_Con_Energia = df_proyectosUnidos.join(
    df_energia_agrupada,
    on=["empresa_id", "tipo_energia"],
    how="left"
)

dfProyectos_Con_Energia.show(10)

+----------+---------------+-----------+--------------------+-------------------+--------------+---------------+---------------+----------------------+
|empresa_id|   tipo_energia|proyecto_id|capacidad_generacion|reduccion_emisiones|costo_proyecto|estado_proyecto|total_inversion|energia_generada_total|
+----------+---------------+-----------+--------------------+-------------------+--------------+---------------+---------------+----------------------+
| EMP016520|          Solar| PROJ000000|             9131.06|            2270.88|         26.03|         Activo|          26.03|               9131.06|
| EMP012923|          Solar| PROJ000001|             2164.32|             4886.1|        364.36|     Finalizado|         386.61|               3645.25|
| EMP023136|        Biomasa| PROJ000002|             2374.24|              446.3|        190.17|  En desarrollo|         190.17|               2374.24|
| EMP004436|    GeotÃ©rmica| PROJ000003|             4065.11|            4465.83|       

●	emisiones_evitadas_total
Total de emisiones evitadas por los proyectos energéticos de una empresa dentro de un mismo tipo de energía. Representa la suma de todas las reducciones de emisiones registradas para estos proyectos.


In [45]:
df_emisiones_evitadas = dataProyectos_transformado.groupBy("empresa_id","tipo_energia").agg(
    round(_sum("reduccion_emisiones"),2).alias("emisiones_evitadas_total")
)

 Se reemplaza los valores NULL por 0.0 de inmediato para proteger los datos

In [46]:
dfProyectos_Con_Emisiones_Evitadas = dfProyectos_Con_Energia.join(
    df_emisiones_evitadas,
    on=["empresa_id", "tipo_energia"],
    how="left"
)

In [47]:
dfProyectos_Con_Emisiones_Evitadas = dfProyectos_Con_Emisiones_Evitadas.fillna(
    {"emisiones_evitadas_total": 0.0})

Se valida el resultado final limpio

In [48]:
dfProyectos_Con_Emisiones_Evitadas.show(20)

+----------+---------------+-----------+--------------------+-------------------+--------------+---------------+---------------+----------------------+------------------------+
|empresa_id|   tipo_energia|proyecto_id|capacidad_generacion|reduccion_emisiones|costo_proyecto|estado_proyecto|total_inversion|energia_generada_total|emisiones_evitadas_total|
+----------+---------------+-----------+--------------------+-------------------+--------------+---------------+---------------+----------------------+------------------------+
| EMP016520|          Solar| PROJ000000|             9131.06|            2270.88|         26.03|         Activo|          26.03|               9131.06|                 2270.88|
| EMP012923|          Solar| PROJ000001|             2164.32|             4886.1|        364.36|     Finalizado|         386.61|               3645.25|                 6823.68|
| EMP023136|        Biomasa| PROJ000002|             2374.24|              446.3|        190.17|  En desarrollo|   

●	retorno_estimado
Indicador que refleja la relación entre la inversión realizada y el impacto ambiental logrado.
Se calcula dividiendo la inversión total en proyectos del tipo de energía correspondiente entre el total de emisiones evitadas por esos mismos proyectos.


Se Calcula el retorno estimado dividiendo la inversión entre las emisiones evitadas

In [49]:
dfProyectos_Final = dfProyectos_Con_Emisiones_Evitadas.withColumn(
    "retorno_estimado",
    when(
        col("emisiones_evitadas_total") > 0,
        round(col("total_inversion") / col("emisiones_evitadas_total"), 2)
    ).otherwise(0.0)
)

In [50]:
dfProyectos_Final.select(
    "empresa_id",
    "tipo_energia",
    "total_inversion",
    "emisiones_evitadas_total",
    "retorno_estimado"
).orderBy("empresa_id", ascending = True).show(20)

+----------+---------------+---------------+------------------------+----------------+
|empresa_id|   tipo_energia|total_inversion|emisiones_evitadas_total|retorno_estimado|
+----------+---------------+---------------+------------------------+----------------+
| EMP000000|HidroelÃ©ctrica|         301.21|                  224.96|            1.34|
| EMP000000|        EÃ³lica|         375.11|                  606.28|            0.62|
| EMP000001|        Biomasa|           4.58|                 1565.71|             0.0|
| EMP000004|        Biomasa|         249.27|                 3963.83|            0.06|
| EMP000005|          Solar|         130.73|                  1965.9|            0.07|
| EMP000005|    GeotÃ©rmica|         151.59|                 1551.77|             0.1|
| EMP000007|        Biomasa|         275.83|                  3201.3|            0.09|
| EMP000007|        EÃ³lica|         421.88|                   60.84|            6.93|
| EMP000008|          Solar|          247.0

In [51]:
dfProyectos_Final.coalesce(1).write.mode("overwrite").options(header="True", sep=",").csv("tabla_salida_2")

archivos_part = glob.glob("tabla_salida_2/part-*.csv")

if archivos_part:

    archivo_origen = archivos_part[0]
    archivo_destino = "tabla_salida_2.csv"
    os.rename(archivo_origen, archivo_destino)
    print(f"¡Listo! Tu archivo se guardó y renombró correctamente como: {archivo_destino}")
else:
    print("No se encontró el archivo part dentro de la carpeta.")

¡Listo! Tu archivo se guardó y renombró correctamente como: tabla_salida_2.csv
